# SpMV 算子开发实验

## 6.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info


<a id="s-01-02_backend_interface_and_execution_model"></a>


## 6.2 先运行 FP32 设备算子

先看每个核怎样分行，再编译运行并与 CPU reference 比较。


### 数据准备与计算

所有实现使用 `ISpmvBackend` 接口：`prepare()` 接收不变的 CSR 矩阵，`run()` 接收本次向量 x 并生成 y。矩阵只需准备一次，后续可换输入向量重复计算。

`AscendCSpmvBackend` 使用 ACL 分配设备内存，通过 RTC 在运行时编译、加载和启动 FP32 Kernel。`npu_spmv*.cpp` 与 `npu_spmv_context.cpp` 中的低精度和数据复用原型在主机 CPU 上计算。


### 每个核计算哪些行

设备源码位于 `src/ascend_spmv/kernels/spmv_fp32_kernel.cpp`。`GetBlockIdx()` 是当前核编号，`GetBlockNum()` 是启动的核数。程序先求每核的行数，并向上取整到 16 行，再计算 `first` 和 `last`。

FP32 的 16 个元素占 64 字节。按这个边界分段，使不同核写入不同的输出缓存行；最后一段截到实际行数。

```cpp
for (int64_t row = first; row < last; ++row) {
        float sum = 0.0F;
        const int32_t begin = rowPtrGm.GetValue(static_cast<int32_t>(row));
        const int32_t end = rowPtrGm.GetValue(static_cast<int32_t>(row + 1));
        for (int32_t index = begin; index < end; ++index) {
            sum += valuesGm.GetValue(index) * xGm.GetValue(colIdxGm.GetValue(index));
        }
        yGm.SetValue(static_cast<int32_t>(row), sum);
    }
```

Host 侧的 `ascendc_spmv.cpp` 负责设备内存、RTC 编译、启动和同步。


### 调用与计时

`spmv_benchmark.cpp` 先调用一次 `prepare()`，再进行预热和重复计算，汇总各次 `run()` 的时间，并与 CPU 参考结果比较误差。

- `host_prototype_*`：CPU 原型的阶段时间。
- `ascendc_transfer_in_ms`、`ascendc_transfer_out_ms`：设备输入、输出传输时间。
- `ascendc_launch_to_complete_ms`：提交 Kernel 到 Stream 同步完成的时间。

统一接口让测试程序可以调用不同实现。比较结果时，结合运行日志中的 `Actual Backend` 确认执行位置。


### 构建与快速运行

加载 CANN 环境后执行下面的命令，构建 Ascend C RTC 版本。成功运行时日志显示 `Actual Backend=Ascend C RTC`，并生成结果 CSV。


In [ ]:
%%bash
set -e
cd src/ascend_spmv
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j
# 学生修改：--matrix 选择输入矩阵；--csv 指定本次结果文件。
./build/bin/spmv_benchmark --matrix U1 --warmup 1 --repeat 3 --csv results/course_quick_check.csv


### 课后实践

在源码中找到 FP32 的设备调用和低精度原型的主机循环，说明它们分别在哪里计算。参考答案见 `answer/06.02_answer.md`。


<a id="s-01-03_precision_and_partition"></a>


## 6.3 BF16 存储与分区（选读）

下面在 CPU 原型中观察低精度存储和分区的取舍。


### 精度与分区

SpMV 每读取一个非零元只做少量乘加，因此数据读取往往占较大开销。FP16/BF16 用 2 字节保存数值，比 FP32 的 4 字节更省空间。本实验 CPU 原型在乘法前恢复为 FP32，并用 FP32 累加；存储时丢失的精度仍会影响结果，需要记录误差。

各行非零元数量可能不同，均分行数不一定均分工作量。`build_nnz_aware_partitions` 根据累计非零元数量选择连续行边界，尽量均衡各分区。


### 观察与解释

FP32 values 每个占 4 字节，BF16 占 2 字节，但列索引和行指针仍占 4 字节。
若有 R 行、Z 个非零元，FP32 CSR 约占 `4(R+1)+8Z` 字节，BF16 约占 `4(R+1)+6Z` 字节。
当每行约 10 个非零元时，两者之比约为 `84/64=1.3125`，不是 2。

本节混合精度路径用于观察精度与存储的取舍，记录实际误差，不设置统一 PASS 阈值。FP32 设备路径的相对误差应小于 `1e-6`。

练习：用上式计算 U1 的两种存储量，再解释为什么压缩 values 不一定让总运行时间同比下降。

[参考答案](answer/06.03_answer.md)


压缩比 `compression_ratio` 按“FP32 CSR 字节数 ÷ 低精度 CSR 字节数”计算。例如 84 字节压到 64 字节，压缩比为 1.3125，节省约 23.8% 空间。数值越大，节省空间越多。


<a id="s-01-04_persistent_context"></a>


## 6.4 矩阵复用（选读）

先准备不变的矩阵与缓冲区，再复用它们计算多个输入向量。


### 矩阵只准备一次

GMRES 会在矩阵 A 不变时反复调用 SpMV。`initialize_bf16_autotuned()` 先转换 BF16、试测候选分区并准备缓冲区，后续 `run_bf16()` 复用这些数据。

本节在 CPU 原型中观察这种复用方式。重点是把一次性准备与每次计算分开，计算重复调用时能省下多少准备工作。


普通 BF16 原型使用串行分区循环，persistent 原型同时启用了 OpenMP。因此两者的时间差还包含并行方式的变化。分析“只准备一次”的价值时，用同一个 persistent 实现的 `cold_start + N × warm total` 估算，不把全部加速归因于复用。

`candidates` 表示 CPU 原型尝试的分区数量，不等于线程数。列表里的 `handle_.ai_core_count` 是沿用的参数名，此处不表示原型运行在 NPU。


### 时间字段包含哪些工作

- `cold_start`：一次性初始化，包含格式转换、分区试测和缓冲区准备。
- `warm_kernel`：复用这些数据后的计算时间。
- `warm total`：每次输入转换、计算和输出复制的总时间。
- `amortized_runtime`：`warm total + cold_start / repeat`，用于估算重复调用时的平均成本。

首次完成计算的时间还要把第一次 run 加到初始化上。矩阵只用一次和重复用 1000 次，适合的方案可能不同。

这些字段对应 Host 原型；真实 FP32 Device 路径在 CSV 中使用 `ascendc_*` 字段。


### 练习

先理解上面的公式；读取 [6.5 节的结果](06_spmv.ipynb#s-01-05_build_validation_and_performance)后，回到本题，用 persistent 原型的初始化时间和每次总时间，分别计算重复 1、10、1000 次时的总成本。指出哪部分只付一次，哪部分每次都要付。

[参考答案](answer/06.04_answer.md)


<a id="s-01-05_build_validation_and_performance"></a>


## 6.5 读取结果并分析

完整 CSV 会保留所有实现的字段。下面只列出本节需要的设备 FP32 和 CPU BF16 数据。


### 设备结果与 CPU 原型

FP32 设备相对误差应小于 `1e-6`。BF16 原型用于观察精度与成本，分析时注明 CPU 执行位置。


In [ ]:
import csv
from pathlib import Path

with Path("src/ascend_spmv/results/course_quick_check.csv").open() as stream:
    row = next(csv.DictReader(stream))
print("实现、位置 | 初始化 ms | 每次总时间 ms | 相对误差")
print("FP32、NPU |", row["ascendc_init_ms"], row["ascendc_total_ms"], row["ascendc_error"])
print("BF16、CPU |", row["host_prototype_bf16_init_ms"],
      row["host_prototype_bf16_total_ms"], row["host_prototype_bf16_mixed_error"])
print("BF16 persistent、CPU/OpenMP |", row["host_prototype_bf16_persistent_init_ms"],
      row["host_prototype_bf16_persistent_total_ms"], row["host_prototype_bf16_persistent_error"])


### 分析本次结果

选择 U1、U2，比较 CPU 单线程、OpenMP、Ascend C FP32 和 CPU 混合精度原型。记录各自的初始化、计算、传输时间与误差，解释矩阵规模和固定开销对结果的影响。

数据复用实验中，分别计算调用 1 次和多次时的总成本。CPU 原型与 NPU 分别列出，避免把执行位置不同的结果混为一类。

### 课后实践

用本次 CSV 说明：哪种实现适合本次输入？改变矩阵规模后，结论是否变化？参考答案见 `answer/06.05_answer.md`。

旧版 CPU 原型数据保存在 `src/ascend_spmv/README.md`，可选读其分析方法。
